# 📐 Linear Algebra for Data Science
### eMobilis · Data Science & AI

> **How to use this notebook:**  
> Read the markdown notes first → run the code → tweak numbers and re-run.  
> Every code cell has comments explaining *what*, *why*, and *what to expect*.

---

| Module | Topic |
|--------|-------|
| 0 | Bridge from Statistics to Linear Algebra |
| 1 | Vectors — The Language of a Single Observation |
| 2 | Matrices — Your Entire Dataset as One Object |
| 3 | The Determinant — Can This Matrix Be Reversed? |
| 4 | Matrix Inverse — Undoing a Transformation |
| 5 | Solving Linear Systems (AX = B) |
| 6 | Rank, Linear Independence & Null Space |
| 7 | Eigenvalues & Eigenvectors — The Natural Directions |
| 8 | Matrix Decompositions — LU, QR, SVD |
| 9 | Real-World ML Pipelines |

In [ ]:
# ============================================================
# RUN THIS CELL FIRST — imports used throughout the notebook
# ============================================================
import numpy as np
import pandas as pd
from scipy import linalg
from scipy.linalg import null_space, lu, svd as scipy_svd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity

print("✓ All imports ready")
print(f"  numpy  {np.__version__}")
print(f"  pandas {pd.__version__}")

---
## Module 0 — Bridge from Statistics to Linear Algebra

### The problem with stats tools at scale

- Stats gave us tools for **1–2 variables** at a time: one distribution, one t-test, one covariance
- Real data: 200 matatus × 50 features = **10,000 numbers**
- 500 features → **124,750 correlation pairs** — can't loop through that
- We need a language that handles **all variables at once** → **linear algebra**

### You already used linear algebra (without calling it that)

| Stats concept you know | Linear algebra twin | Module |
|---|---|---|
| One matatu's data row | **Vector** | 1 |
| Entire fleet CSV / DataFrame | **Matrix** | 2 |
| Cov(X, Y) computed by hand | One entry in the **covariance matrix** | 2.6 |
| Pearson $r = Cov / (\sigma_X \sigma_Y)$ | **Normalised dot product** of two vectors | 1.4 |
| Finding regression line | Solve the **matrix equation** $AX = B$ | 5 |
| PCA directions | **Eigenvectors** of the covariance matrix | 7 |

> **Key insight:** Statistics gives you the questions. Linear algebra is the engine that answers them at scale.

### Why matrices are astronomically faster than for-loops

| Factor | What it means |
|---|---|
| **No Python interpreter overhead** | NumPy sends the whole array to compiled C in one call — no per-element type-checking |
| **SIMD instructions** | Modern CPUs (SSE, AVX) multiply **4–8 floats in a single clock cycle** |
| **Cache-friendly memory** | NumPy stores data contiguously — CPU loads whole blocks into fast cache |
| **BLAS libraries** | Hand-optimised C/Fortran routines (DGEMM for matrix multiply) refined over 40 years |
| **GPU parallelism** | GPUs have thousands of cores — PyTorch/TensorFlow run all matrix ops simultaneously |

> **Career rule:** if you find yourself writing a `for` loop over rows or columns of data, ask whether a matrix operation can replace it. In production, that swap can turn a 2-hour job into 3 minutes.

In [ ]:
# ============================================================
# SPEED DEMO: for-loop vs NumPy
# ============================================================
import time, numpy as np

np.random.seed(42)
n = 10_000
trips      = np.random.randint(5, 20, size=n)
fare_rates = np.random.uniform(1.5, 3.0, size=n)

# --- Method 1: pure Python for-loop ---
start = time.perf_counter()
revenues_loop = []
for i in range(n):
    revenues_loop.append(trips[i] * fare_rates[i])   # Python steps through each one
loop_ms = (time.perf_counter() - start) * 1000

# --- Method 2: NumPy vectorised ---
start = time.perf_counter()
revenues_np = trips * fare_rates    # ONE call at the C level
np_ms = (time.perf_counter() - start) * 1000

print(f"For-loop time : {loop_ms:.2f} ms")
print(f"NumPy time    : {np_ms:.4f} ms")
print(f"Speedup       : {loop_ms/np_ms:.0f}x faster")
print(f"Same result?  : {np.allclose(revenues_loop, revenues_np)}")

# Same idea, larger scale: 1 million operations
n_big = 1_000_000
big = np.random.rand(n_big)
start = time.perf_counter(); _ = big * 2.5; t = time.perf_counter() - start
print(f"\n1M element multiply: {t*1000:.3f} ms  (~{1/t/1e6:.0f} million ops/sec)")

---
## Module 1 — Vectors: The Language of a Single Observation

### 1.1 What Is a Vector?

- An **ordered list of numbers** — order matters: `[3, 4] ≠ [4, 3]`
- Each number = a **component** (also: element, entry, feature value)
- **Dimension** = number of components — just count them
- In data science: **one vector = one row of your dataset**

```
matatu_1 = [trips=8,  fare=12,  fuel=3.5,  passengers=120]   ← 4D vector
matatu_2 = [trips=10, fare=15,  fuel=4.1,  passengers=145]   ← another 4D vector
```

> ⚠️ **Dimension ≠ physical space.** A 500D vector is just 500 numbers in a row. Your brain can't picture it; NumPy computes it just fine.

### Row vs column vectors
- **Row vector:** `[3, 4, 5]` → shape `(3,)`
- **Column vector:** same numbers, written vertically → shape `(3, 1)`  
- Orientation only matters when multiplying matrices — same data either way

In [ ]:
import numpy as np

# ============================================================
# 1.1  VECTORS IN NUMPY
# ============================================================

# One matatu's daily stats = one vector
matatu_1 = np.array([8, 12.0, 3.5, 120])    # [trips, fare_k, fuel_L, passengers]
matatu_2 = np.array([10, 15.0, 4.1, 145])

print("matatu_1 :", matatu_1)
print("Dimension:", len(matatu_1))            # 4 components → 4-dimensional vector
print("Shape    :", matatu_1.shape)           # (4,)

# Accessing individual components
print("\nTrips      (index 0):", matatu_1[0])
print("Fare       (index 1):", matatu_1[1])
print("Fuel       (index 2):", matatu_1[2])
print("Passengers (index 3):", matatu_1[3])

# Row vs column vector — same numbers, different orientation
row_vec = np.array([3.0, 4.0, 5.0])           # shape (3,)
col_vec = row_vec.reshape(-1, 1)               # shape (3, 1)

print("\nRow vector shape:", row_vec.shape)
print("Col vector shape:", col_vec.shape)
print("Same data? True — orientation only matters in matrix multiply")

### 1.2 Vector Operations

#### Addition — combine two observations component-wise
$$\mathbf{u} + \mathbf{v} = [u_1+v_1,\ u_2+v_2,\ \ldots,\ u_n+v_n]$$
- Vectors **must have the same dimension**
- Real use: morning stats + afternoon stats = day total

#### Scalar multiplication — stretch or shrink everything
$$c \cdot \mathbf{v} = [c \cdot v_1,\ c \cdot v_2,\ \ldots,\ c \cdot v_n]$$
- $c > 1$ stretches · $0 < c < 1$ shrinks · $c = -1$ reverses direction

#### Magnitude (L2 norm) — how "big" is the vector?
$$\|\mathbf{v}\| = \sqrt{v_1^2 + v_2^2 + \cdots + v_n^2}$$
- Pythagoras extended to $n$ dimensions
- **ML use:** features with large magnitudes dominate models → we normalise to fix this

#### Unit vector — pure direction, length = 1
$$\hat{\mathbf{v}} = \frac{\mathbf{v}}{\|\mathbf{v}\|}$$
- Called **normalisation** — essential before computing any similarity metric

In [ ]:
import numpy as np

# ============================================================
# 1.2  VECTOR OPERATIONS
# ============================================================

morning   = np.array([6, 9.0])   # [trips, fare_k] — morning run
afternoon = np.array([4, 7.0])   # afternoon run

# --- Addition ---
day_total = morning + afternoon
print("Morning + Afternoon =", day_total)    # [10.  16.] — day totals

# --- Scalar multiplication ---
# Fare prices increase 20%
scaled = morning * 1.2
print("After 20% fare hike:", scaled)        # trips unchanged, fare scaled

# Negative scalar — reverses direction (used in gradient descent: w -= lr * grad)
reversed_v = morning * -1
print("Reversed:", reversed_v)              # [-6, -9]

# --- Magnitude ---
v = np.array([3.0, 4.0])
magnitude_manual = np.sqrt(v[0]**2 + v[1]**2)   # Pythagoras
magnitude_numpy  = np.linalg.norm(v)
print(f"\n||[3,4]|| = {magnitude_manual}  (manual: sqrt(9+16)=5)")
print(f"||[3,4]|| = {magnitude_numpy}  (numpy)")

# 3D: same formula, same result as 2D for these numbers
v3d = np.array([3.0, 4.0, 0.0])
print(f"||[3,4,0]|| = {np.linalg.norm(v3d)}")   # still 5.0

# --- Unit vector (normalisation) ---
unit_v = v / np.linalg.norm(v)
print("\nUnit vector:", unit_v)                  # [0.6, 0.8]
print("Length of unit vector:", np.linalg.norm(unit_v))   # always 1.0

# Real-world: normalise a matatu's full feature vector
matatu = np.array([8.0, 12000.0, 3.5, 120.0])  # mixed scales!
matatu_norm = matatu / np.linalg.norm(matatu)
print("\nOriginal magnitude :", np.linalg.norm(matatu).round(2))
print("Normalised magnitude:", np.linalg.norm(matatu_norm))   # exactly 1.0

### 1.3 The Dot Product — The Most Important Operation in ML

#### Algebraic definition — how to compute it
$$\mathbf{u} \cdot \mathbf{v} = u_1 v_1 + u_2 v_2 + \cdots + u_n v_n$$
- Multiply **matching components**, then **sum** — result is a **single number (scalar)**
- Example: $[1,2,3] \cdot [4,5,6] = 4 + 10 + 18 = 32$

#### Geometric meaning — what is it measuring?
$$\mathbf{u} \cdot \mathbf{v} = \|\mathbf{u}\| \cdot \|\mathbf{v}\| \cdot \cos\theta$$

| Angle $\theta$ | $\cos\theta$ | Dot product | Meaning |
|---|---|---|---|
| 0° (same direction) | +1 | large positive | Maximum agreement |
| 90° (perpendicular) | 0 | 0 | No shared direction |
| 180° (opposite) | −1 | large negative | Maximum disagreement |

> **Key insight:** The dot product measures how much two vectors *agree in direction*.  
> In ML — when a model "scores" an input, that score **is** a dot product of features × weights.

#### Where it appears in every ML model

| Model | The dot product |
|---|---|
| Linear regression | `prediction = w · x` |
| Logistic regression | `sigmoid(w · x)` |
| Each neural network neuron | `activation(w · x + b)` |
| Recommendation system | `user_vec · item_vec` = preference score |
| Word similarity (NLP) | cosine of word embedding vectors |

In [ ]:
import numpy as np

# ============================================================
# 1.3  THE DOT PRODUCT
# ============================================================

# --- Three classic cases ---
u_same = np.array([1.0, 1.0]);  v_same = np.array([2.0, 2.0])  # same direction
u_opp  = np.array([1.0, 1.0]);  v_opp  = np.array([-2.0,-2.0]) # opposite
u_perp = np.array([1.0, 0.0]);  v_perp = np.array([0.0, 1.0])  # perpendicular

print("Same direction    np.dot:", np.dot(u_same, v_same))  # 4  → positive
print("Opposite direction np.dot:", np.dot(u_opp, v_opp))  # -4 → negative
print("Perpendicular      np.dot:", np.dot(u_perp, v_perp)) # 0  → zero

# --- Manual step-by-step ---
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 5.0, 6.0])
manual = u[0]*v[0] + u[1]*v[1] + u[2]*v[2]
print(f"\n[1,2,3] · [4,5,6] = 1×4 + 2×5 + 3×6 = {manual}")
print(f"np.dot result      = {np.dot(u, v)}")

# --- Real-world: dot product as a prediction score ---
# Matatu features: trips, passengers, fare
features = np.array([10.0, 145.0, 15.0])

# "Learned" weights: how much each feature predicts daily revenue
weights = np.array([0.5, 0.02, 1.0])

score = np.dot(features, weights)
print("\n=== Revenue Prediction (dot product) ===")
print(f"  trips × 0.50       = {features[0]*weights[0]:.2f}")
print(f"  passengers × 0.02  = {features[1]*weights[1]:.2f}")
print(f"  fare × 1.00        = {features[2]*weights[2]:.2f}")
print(f"  Total score        = {score:.2f}")
print("This is EXACTLY what LinearRegression.predict() does for each sample")

# --- Angle between two vectors from the dot product ---
theta_rad = np.arccos(np.dot(u_same, v_same) /
                      (np.linalg.norm(u_same) * np.linalg.norm(v_same)))
print(f"\nAngle between [1,1] and [2,2]: {np.degrees(theta_rad):.1f}°")  # 0°

### 1.4 Cosine Similarity — Dot Product Meets Pearson r

$$\text{cosine\_sim}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \cdot \|\mathbf{v}\|}$$

- Dot product divided by both magnitudes → **strips out scale**, keeps direction
- Always in $[-1, +1]$ — exactly like Pearson $r$
- $+1$ = identical direction · $0$ = no relationship · $-1$ = opposite

> **Connection to Pearson r you already know:**  
> $r = \frac{Cov(X,Y)}{\sigma_X \sigma_Y}$ is the cosine similarity of the **deviation vectors** $(X - \bar{X})$ and $(Y - \bar{Y})$.  
> Linear algebra was underneath your stats formula all along — just not labelled.

> ⚠️ **Always normalise before similarity.** Raw dot products are dominated by whichever feature has the largest numbers. Cosine similarity strips that out, giving you **pure directional similarity**.

**Used in:** NLP word embeddings · recommendation systems · image search · customer segmentation

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# 1.4  COSINE SIMILARITY
# ============================================================

# Two matatus — usage patterns
matatu_a = np.array([170.0, 12000.0, 120.0])   # [trips, fare_KES, passengers]
matatu_b = np.array([175.0, 12500.0, 125.0])   # similar, slightly bigger
matatu_c = np.array([8.0,   3000.0,  40.0])    # very different (low activity)

# --- Raw dot product: misleading ---
raw_ab = np.dot(matatu_a, matatu_b)
raw_ac = np.dot(matatu_a, matatu_c)
print("Raw dot product A·B:", f"{raw_ab:>15,.0f}   ← dominated by fare (12000)")
print("Raw dot product A·C:", f"{raw_ac:>15,.0f}   ← also large, can't compare")

# --- Cosine similarity: correct ---
def cosine_sim(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

sim_ab = cosine_sim(matatu_a, matatu_b)
sim_ac = cosine_sim(matatu_a, matatu_c)
print(f"\nCosine sim  A vs B (similar)  : {sim_ab:.4f}")   # close to 1
print(f"Cosine sim  A vs C (different) : {sim_ac:.4f}")   # lower

# --- sklearn convenience ---
X_stacked = np.array([matatu_a, matatu_b, matatu_c])
sim_matrix = cosine_similarity(X_stacked)
print("\nFull 3×3 similarity matrix:")
print(np.round(sim_matrix, 4))

# --- Connection to Pearson r ---
# Pearson r IS cosine similarity of deviation vectors
x = np.array([8.0, 10, 12, 14, 16])
y = np.array([12.0, 15, 16, 20, 22])

# Method 1: Pearson r (from stats)
r_pearson = np.corrcoef(x, y)[0, 1]

# Method 2: cosine similarity of deviation vectors (linear algebra)
x_dev = x - x.mean()
y_dev = y - y.mean()
r_cosine = cosine_sim(x_dev, y_dev)

print(f"\nPearson r (np.corrcoef) : {r_pearson:.6f}")
print(f"Cosine sim of deviations: {r_cosine:.6f}")
print(f"Same thing?             : {np.isclose(r_pearson, r_cosine)}")

### 1.5 Linear Independence — Does Each Vector Add New Information?

**Question:** Can you build one vector from the others using scaling + addition?

- **YES → Linearly dependent** — that vector is redundant; it adds no new direction
- **NO → Linearly independent** — it points in a genuinely new direction

#### Concrete example
```
A = [4, 8]   → 4 trips, 8k fare
B = [2, 4]   → B = 0.5 × A  ← DEPENDENT (B tells you nothing A doesn't)
C = [3, 7]   → cannot be scaled from A  ← INDEPENDENT (new direction)
```

#### Formal definition
$\mathbf{v}_1, \ldots, \mathbf{v}_k$ are **linearly independent** if:
$$c_1\mathbf{v}_1 + c_2\mathbf{v}_2 + \cdots + c_k\mathbf{v}_k = \mathbf{0} \implies c_1 = c_2 = \cdots = c_k = 0$$

The only way to get zero is with all-zero weights. Any non-zero weights that produce zero → dependent.

> **In data science:** Perfectly correlated features ($r = 1$) are linearly dependent.  
> Keeping both doesn't add information but **breaks matrix inversion** → "Singular matrix" error.

In [ ]:
import numpy as np

# ============================================================
# 1.5  LINEAR INDEPENDENCE
# ============================================================

A = np.array([4.0, 8.0])
B = np.array([2.0, 4.0])   # B = 0.5 * A → DEPENDENT
C = np.array([3.0, 7.0])   # cannot be scaled from A → INDEPENDENT

# Quick check: if B = k*A, then B/A should be constant
print("B / A =", B / A)   # [0.5, 0.5] — same ratio → DEPENDENT
print("C / A =", C / A)   # [0.75, 0.875] — different → INDEPENDENT

# --- Dataset with a redundant (dependent) column ---
trips     = np.array([8.0, 10, 12, 14, 16])
fare      = np.array([12.0, 15, 16, 20, 22])
fare_usd  = fare / 130     # exact linear multiple of fare → DEPENDENT
dist      = np.array([25.0, 30, 33, 40, 45])   # genuinely new info

X = np.column_stack([trips, fare, fare_usd, dist])
print("\nDataset shape:", X.shape)
print("Rank:", np.linalg.matrix_rank(X))   # 3, not 4! fare_usd is redundant

# --- The crash: trying to invert a singular matrix ---
XtX = X.T @ X   # 4×4 matrix, but rank 3 → singular

try:
    beta = np.linalg.inv(XtX) @ X.T @ np.random.rand(5)
    print("Inverse computed (shouldn't happen)")
except np.linalg.LinAlgError as e:
    print(f"\n❌ Error: {e}")
    print("   Cause: 'fare_usd' is a linear multiple of 'fare' → singular matrix")
    print("   Fix:   drop redundant feature before fitting model")

---
## Module 2 — Matrices: Your Entire Dataset as One Object

### 2.1 What Is a Matrix?

- A **rectangular grid of numbers** — $m$ rows × $n$ columns → called **$m \times n$ matrix**
- In data science: **your matrix IS your dataset**
  - Each **row** = one observation (one matatu, one student, one transaction)
  - Each **column** = one feature (trips, fare, fuel, passengers)
- 200 matatus × 5 features → **200 × 5 matrix**
- Every `pd.read_csv()` creates a matrix. DataFrames **are** matrices with labels.

$$A = \begin{bmatrix} a_{11} & a_{12} & \cdots & a_{1n} \\ a_{21} & a_{22} & \cdots & a_{2n} \\ \vdots & & \ddots & \vdots \\ a_{m1} & a_{m2} & \cdots & a_{mn} \end{bmatrix}$$

- $a_{ij}$ = element at row $i$, column $j$ (1-indexed in maths, 0-indexed in Python)

### 2.2 Memory Efficiency

- Without matrix: 200 separate lists → **scattered in memory** → CPU cache misses
- NumPy matrix: all numbers in **one contiguous block** — the CPU loads chunks efficiently
- `shape`, `dtype`, memory address stored **once** (not per row)
- Slicing a row or column is $O(1)$ — just a pointer, no copying

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# 2.1 & 2.2  THE DATASET AS A MATRIX
# ============================================================

# 5 matatus × 4 features: [trips, fare_k, fuel_L, passengers]
fleet = np.array([
    [8,  12.0, 3.5, 120],
    [10, 15.0, 4.1, 145],
    [12, 16.0, 4.4, 160],
    [14, 20.0, 5.0, 180],
    [16, 22.0, 5.3, 200],
])

print("=== Matrix properties ===")
print("Shape    :", fleet.shape)           # (5, 4)
print("Rows     :", fleet.shape[0])        # 5 matatus
print("Columns  :", fleet.shape[1])        # 4 features
print("Total    :", fleet.size, "numbers")

# Accessing data
print("\n=== Accessing elements ===")
print("Matatu 1 (row 0)            :", fleet[0])
print("All trips (column 0)        :", fleet[:, 0])
print("Top 3 matatus               :\n", fleet[:3])
print("Fare of matatu 3 [2,1]      :", fleet[2, 1])

# The same data as a DataFrame — matrix underneath
df = pd.DataFrame(fleet, columns=["trips", "fare_k", "fuel_L", "passengers"])
print("\n=== As DataFrame ===")
print(df)
print("\nType of df.values:", type(df.values))   # it's a numpy ndarray
print("Same data:", np.array_equal(fleet, df.values))

# Operations on ALL data at once
print("\n=== Column operations (all features simultaneously) ===")
print("Column means :", fleet.mean(axis=0))
print("Column stds  :", fleet.std(axis=0, ddof=1).round(3))

### 2.3 Matrix Addition & Scalar Multiplication

**Addition:** same shape required → add entry-by-entry
$$(A + B)_{ij} = a_{ij} + b_{ij}$$

**Scalar multiplication:** multiply every entry by a constant $c$

### 2.4 Matrix Multiplication — The Engine of All ML

**The rule:** Element $(i,j)$ of the result = **dot product** of row $i$ of $A$ with column $j$ of $B$

$$C = A \cdot B \quad\quad c_{ij} = \sum_k a_{ik} b_{kj}$$

**Dimension rule:** $(m \times n) \cdot (n \times p) = (m \times p)$ — inner dimensions must match and are "consumed"

$$\underbrace{(200 \times 5)}_{\text{dataset}} \cdot \underbrace{(5 \times 1)}_{\text{weights}} = \underbrace{(200 \times 1)}_{200 \text{ predictions}}$$

| | Code | Shape rule | Use when |
|---|---|---|---|
| **Element-wise** | `A * B` | Must be same shape | Scaling, feature-wise operations |
| **Matrix multiply** | `A @ B` | Cols of A = Rows of B | Predictions, transformations, neural layers |

> ⚠️ **NOT commutative:** $A \cdot B \neq B \cdot A$ in general. Order matters.

In [ ]:
import numpy as np

# ============================================================
# 2.3 & 2.4  MATRIX OPERATIONS
# ============================================================

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("=== Addition ===")
print(A + B)            # [[6,8],[10,12]] -- entry by entry

print("\n=== Scalar multiply (A × 3) ===")
print(3 * A)            # [[3,6],[9,12]]

# --- Matrix multiply: step by step ---
print("\n=== Matrix Multiply A @ B ===")
# C[0,0] = row0·col0 = 1×5 + 2×7 = 19
# C[0,1] = row0·col1 = 1×6 + 2×8 = 22
# C[1,0] = row1·col0 = 3×5 + 4×7 = 43
# C[1,1] = row1·col1 = 3×6 + 4×8 = 50
C = A @ B
print(C)

# --- Element-wise vs matrix multiply: completely different results ---
print("\n=== ELEMENT-WISE (A * B) vs MATRIX MULTIPLY (A @ B) ===")
print("A * B  (element-wise) :\n", A * B)    # [[5,12],[21,32]]
print("A @ B  (matrix mul)   :\n", A @ B)    # [[19,22],[43,50]]

# --- The real use: predicting for ALL matatus at once ---
fleet = np.array([
    [1, 8,  12.0, 3.5, 120],   # [bias, trips, fare_k, fuel, passengers]
    [1, 10, 15.0, 4.1, 145],
    [1, 12, 16.0, 4.4, 160],
    [1, 14, 20.0, 5.0, 180],
    [1, 16, 22.0, 5.3, 200],
])   # shape (5, 5)

weights = np.array([50.0, 2.0, 1.5, -10.0, 0.3])   # shape (5,)

# ONE matrix multiply replaces 5 separate dot products
predictions = fleet @ weights   # shape (5,)
print("\n=== 5 Revenue Predictions via ONE matrix multiply ===")
for i, p in enumerate(predictions):
    print(f"  Matatu {i+1}: {p:.1f}")
print("(sklearn's predict() does exactly this — matrix multiply under the hood)")

# Dimension check
print(f"\nfleet @ weights: ({fleet.shape}) @ ({weights.shape}) = {predictions.shape}")

### 2.5 Transpose — Flipping Rows and Columns

$$(A^\top)_{ij} = A_{ji}$$

- An $m \times n$ matrix becomes $n \times m$ after transposing
- Code: `A.T` in NumPy
- A column vector becomes a row vector (and vice versa)

**Why it's in every ML formula:**

$$\beta = (X^\top X)^{-1} X^\top \mathbf{y}$$

- $X$ is $m \times n$ → $X^\top$ is $n \times m$
- $X^\top X$ is $n \times n$ → **square** → can be inverted
- Every $^\top$ in a formula is making the dimensions align correctly

In [ ]:
import numpy as np

# ============================================================
# 2.5  TRANSPOSE
# ============================================================

A = np.array([[1, 2, 3],
              [4, 5, 6]])   # shape (2, 3)

print("Original A:")
print(A)
print("Shape:", A.shape)         # (2, 3)

print("\nTranspose A.T:")
print(A.T)
print("Shape:", A.T.shape)       # (3, 2) -- swapped

# --- Regression formula shape check ---
# β = (X.T @ X)^{-1} @ X.T @ y
np.random.seed(0)
X = np.random.rand(100, 5)    # 100 matatus, 5 features
y = np.random.rand(100)       # revenue targets

print("\n=== Shapes in β = (X.T @ X)^{-1} @ X.T @ y ===")
print(f"X      shape: {X.shape}")
print(f"X.T    shape: {X.T.shape}")

XtX = X.T @ X
print(f"X.T@X  shape: {XtX.shape}  ← square: can be inverted ✓")

Xty = X.T @ y
print(f"X.T@y  shape: {Xty.shape}  ← right-hand side")

beta = np.linalg.inv(XtX) @ Xty
print(f"beta   shape: {beta.shape}  ← one coefficient per feature")
print(f"beta        : {beta.round(3)}")

### 2.6 Special Matrices

#### Identity matrix $I$ — the "1" for matrices
- Square: 1s on diagonal, 0s everywhere else
- $A \cdot I = I \cdot A = A$ (multiplying by $I$ does nothing)
- Code: `np.eye(n)`

#### Covariance matrix — bridge back to your stats notes

When you computed $Cov(trips, fare)$ by hand → **one number for one pair**.  
The **covariance matrix** computes **all pairs simultaneously:**

$$C = \begin{bmatrix} Var(X_1) & Cov(X_1,X_2) & Cov(X_1,X_3) \\ Cov(X_2,X_1) & Var(X_2) & Cov(X_2,X_3) \\ Cov(X_3,X_1) & Cov(X_3,X_2) & Var(X_3) \end{bmatrix}$$

- **Diagonal** = variance of each feature
- **Off-diagonal** = covariance between each pair
- `df.corr()` = standardised version — all Pearson $r$ values, all pairs, one call

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# 2.6  SPECIAL MATRICES
# ============================================================

# --- Identity matrix ---
I = np.eye(4)
print("4×4 Identity matrix:")
print(I)

A = np.array([[3.0, 1], [2, 4]])
print("\nA @ I == A:", np.allclose(A @ np.eye(2), A))   # True

# --- Covariance matrix ---
fleet = np.array([
    [8,  12.0, 3.5],
    [10, 15.0, 4.1],
    [12, 16.0, 4.4],
    [14, 20.0, 5.0],
    [16, 22.0, 5.3],
])   # 5 matatus, 3 features: trips, fare, fuel

# np.cov expects features as ROWS → transpose first
C = np.cov(fleet.T)
print("\nCovariance matrix (3×3 — all pairs at once):")
labels = ["trips", "fare", "fuel"]
df_cov = pd.DataFrame(C, index=labels, columns=labels)
print(df_cov.round(3))
print("\nDiagonal (variances):", np.diag(C).round(3))

# Correlation matrix: standardised, all values in [-1, 1]
df = pd.DataFrame(fleet, columns=labels)
corr = df.corr()
print("\nCorrelation matrix (all Pearson r at once):")
print(corr.round(3))

# The connection: cov(X,Y) / (std_X * std_Y) == r
cov_trips_fare = C[0, 1]
std_trips = np.sqrt(C[0, 0])
std_fare  = np.sqrt(C[1, 1])
r_manual  = cov_trips_fare / (std_trips * std_fare)
r_pandas  = corr.loc["trips", "fare"]
print(f"\nManual r = Cov / (sd×sd) = {r_manual:.4f}")
print(f"pandas r                 = {r_pandas:.4f}")
print(f"Same? {np.isclose(r_manual, r_pandas)}")

### 2.7 Rank — How Much Real Information Is in the Matrix?

- **Rank** = number of **linearly independent rows (or columns)**
- A matrix with rank $r$ lives in an $r$-dimensional space, even if it has more rows/columns

| Situation | Meaning |
|---|---|
| **Full rank** (rank = min(m,n)) | All rows/cols independent; invertible if square |
| **Rank-deficient** (rank < min(m,n)) | Redundant features; NOT invertible; model may crash |
| **Rank 1** | Every row is a multiple of one row — extreme redundancy |

> **Data science rule:** Before fitting a model, check `np.linalg.matrix_rank(X)`.  
> If rank < n_features, you have redundant columns → drop them first.

In [ ]:
import numpy as np

# ============================================================
# 2.7  RANK
# ============================================================

# Full rank: all rows independent
A_full = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 0]])   # last row is NOT a combination of others
print("Rank (all independent) :", np.linalg.matrix_rank(A_full))   # 3

# Rank-deficient: row 1 = 2 × row 0
A_dep = np.array([[1, 2, 3],
                  [2, 4, 6],    # ← 2 × row 0 — DEPENDENT
                  [0, 1, 2]])
print("Rank (one dep row)     :", np.linalg.matrix_rank(A_dep))    # 2, not 3

# Famous rank-deficient matrix [[1,2,3],[4,5,6],[7,8,9]]
# row2 = row1 + (row1 - row0)
A_classic = np.array([[1,2,3],[4,5,6],[7,8,9]])
print("Rank [[1..9] 3×3]      :", np.linalg.matrix_rank(A_classic)) # 2

# --- Practical dataset check ---
trips     = np.array([[8.0],[10],[12],[14],[16]])
fare      = np.array([[12.0],[15],[16],[20],[22]])
fare_usd  = fare / 130   # same as fare → DEPENDENT
dist      = np.array([[25.0],[30],[33],[40],[45]])

X = np.hstack([trips, fare, fare_usd, dist])
rank = np.linalg.matrix_rank(X)
n_features = X.shape[1]
print(f"\nDataset shape: {X.shape}")
print(f"Rank         : {rank}")
print(f"Redundant    : {n_features - rank} column(s)")
if rank < n_features:
    print("⚠️  Redundant feature detected — remove before modelling")

# Checking all at once with a helper function
def check_rank(X, feature_names=None):
    rank = np.linalg.matrix_rank(X)
    n = X.shape[1]
    status = "✓ Full rank" if rank == n else f"⚠ Rank-deficient ({n-rank} redundant)"
    print(f"  Rank: {rank}/{n} features  →  {status}")

print("\nRank check on our datasets:")
check_rank(np.hstack([trips, fare, dist]), ["trips","fare","dist"])  # full
check_rank(X, ["trips","fare","fare_usd","dist"])                   # deficient

---
## Module 3 — The Determinant: Can This Matrix Be Reversed?

- A **single number** associated with every square matrix
- Geometrically: measures how much the matrix **scales area** (2D) or **volume** (3D)

| $\det(A)$ | Geometric meaning |
|---|---|
| $= 1$ | Area preserved — pure rotation or reflection |
| $= 2$ | Area doubled |
| $= 0.5$ | Area halved |
| $= 0$ | Area **collapses to zero** — information lost — **SINGULAR** |
| $< 0$ | Orientation flipped (mirror image) |

> **Golden rule:** $\det(A) = 0$ → matrix is **singular** → it squashes space down a dimension → information was lost → **no inverse exists**

#### 2×2 formula
$$\det\begin{pmatrix}a & b \\ c & d\end{pmatrix} = ad - bc$$

Worked example:
$$A = \begin{pmatrix}7 & 2 \\ 17 & 5\end{pmatrix} \Rightarrow \det = 7 \times 5 - 2 \times 17 = 35 - 34 = 1$$

Singular example:
$$B = \begin{pmatrix}2 & 4 \\ 1 & 2\end{pmatrix} \Rightarrow \det = 4 - 4 = 0 \quad \text{(row 2 = 0.5 × row 1 — dependent!)}$$

In [ ]:
import numpy as np

# ============================================================
# MODULE 3  THE DETERMINANT
# ============================================================

# --- 2×2 manual formula: det = ad - bc ---
A = np.array([[7.0, 2.0],
              [17.0, 5.0]])

det_manual = A[0,0]*A[1,1] - A[0,1]*A[1,0]   # ad - bc
det_numpy  = np.linalg.det(A)
print(f"det(A) manual: {det_manual}")
print(f"det(A) numpy : {det_numpy:.1f}")
print(f"det ≠ 0 → invertible: {not np.isclose(det_numpy, 0)}")

# --- Singular matrix ---
B = np.array([[2.0, 4.0],
              [1.0, 2.0]])   # row 1 = 0.5 × row 0 → DEPENDENT

det_B = np.linalg.det(B)
print(f"\ndet(B) = {det_B:.1f}  ← singular")
print("row 1 is 0.5 × row 0 → matrix collapses plane to a line")

# --- 3×3 determinant (always use numpy) ---
C = np.array([[1,2,3],[4,5,6],[7,8,9]], dtype=float)
print(f"\ndet(C) = {np.linalg.det(C):.6f}  ← near zero (row 3 ≈ 2×row2 - row1)")

# --- Practical: check before attempting inversion ---
matrices = {
    "Invertible  [[3,1],[2,4]]": np.array([[3.0,1],[2,4]]),
    "Singular    [[2,6],[1,3]]": np.array([[2.0,6],[1,3]]),
    "Near-sing   [[1,2],[2,4.0001]]": np.array([[1.0,2],[2,4.0001]]),
}

print("\n=== Invertibility Check ===")
for name, M in matrices.items():
    det = np.linalg.det(M)
    rank = np.linalg.matrix_rank(M)
    invertible = not np.isclose(det, 0, atol=1e-10)
    print(f"{name}  →  det={det:.4f}  rank={rank}  invertible={invertible}")

---
## Module 4 — Matrix Inverse: Undoing a Transformation

For scalar 5: its inverse is $\frac{1}{5}$ because $5 \times \frac{1}{5} = 1$

For matrix $A$: its inverse $A^{-1}$ satisfies:
$$A \cdot A^{-1} = A^{-1} \cdot A = I$$

- The inverse **undoes** what $A$ did — rotate/stretch then un-rotate/un-stretch
- Only **square matrices** can have an inverse
- Only invertible when $\det(A) \neq 0$

#### 2×2 manual formula
$$A^{-1} = \frac{1}{\det(A)} \begin{pmatrix} d & -b \\ -c & a \end{pmatrix}$$

Steps: swap diagonal → negate off-diagonal → divide by det

#### When no inverse exists
- $\det(A) = 0$ → singular (dependent rows)
- Matrix is not square

> ⚠️ **Ill-conditioned:** even when $\det \neq 0$, if it's very close to 0, the inverse has huge unreliable values. `np.linalg.inv()` won't warn you.

---

### 🚨 Never use `inv()` to solve a linear system

| Approach | Code | Speed | Numerical stability |
|---|---|---|---|
| ❌ Textbook way | `np.linalg.inv(A) @ b` | Slow | Poor (error accumulates) |
| ✅ Correct way | `np.linalg.solve(A, b)` | Fast | Good (LU decomposition) |

In [ ]:
import numpy as np

# ============================================================
# MODULE 4  MATRIX INVERSE
# ============================================================

# --- 2×2 manual inverse ---
A = np.array([[7.0, 2.0],
              [17.0, 5.0]])

a, b, c, d = A[0,0], A[0,1], A[1,0], A[1,1]
det = a*d - b*c

A_inv_manual = (1/det) * np.array([[d, -b],
                                    [-c, a]])
print("Manual inverse:")
print(A_inv_manual)

# Verify: A @ A^{-1} = I
product = A @ A_inv_manual
print("\nA @ A^{-1} (should be identity):")
print(np.round(product, 10))

# --- numpy's inv ---
A_inv_np = np.linalg.inv(A)
print("\nnp.linalg.inv(A) matches manual:", np.allclose(A_inv_manual, A_inv_np))

# --- The right way to solve AX = b ---
b_vec = np.array([9.0, 3.0])

# BAD: inverse then multiply (textbook habit — avoid in code)
X_bad  = np.linalg.inv(A) @ b_vec

# GOOD: solve directly — LU decomposition, no explicit inverse
X_good = np.linalg.solve(A, b_vec)

print("\n=== Solving AX = b ===")
print("Via inv()  :", X_bad)    # [3.6, 0.6] — correct but wasteful
print("Via solve():", X_good)   # [3.6, 0.6] — same, better numerics

# Speed comparison on larger matrices
np.random.seed(1)
N = 500
big_A = np.random.rand(N, N) + N * np.eye(N)   # diagonally dominant → invertible
big_b = np.random.rand(N)

import time
start = time.perf_counter(); _ = np.linalg.inv(big_A) @ big_b; t_inv = time.perf_counter() - start
start = time.perf_counter(); _ = np.linalg.solve(big_A, big_b); t_sol = time.perf_counter() - start

print(f"\n500×500 system — inv() vs solve():")
print(f"  inv()   : {t_inv*1000:.1f} ms")
print(f"  solve() : {t_sol*1000:.1f} ms  ← faster and more stable")

---
## Module 5 — Solving Linear Systems: $AX = B$

Any set of linear equations (no $x^2$, no $\sin x$) can be written as $AX = B$:

$$\begin{cases} 2x + 3y = 9 \\ x - y = 3 \end{cases} \quad\Rightarrow\quad \underbrace{\begin{bmatrix}2&3\\1&-1\end{bmatrix}}_{A}\underbrace{\begin{bmatrix}x\\y\end{bmatrix}}_{X} = \underbrace{\begin{bmatrix}9\\3\end{bmatrix}}_{B}$$

**Where this appears in data science:**
- Regression: find coefficients that fit training data
- Network analysis: balance flows at every node
- Optimisation: find where all gradient components = 0

### The Augmented Matrix $[A\mid B]$
- Attach $B$ to the right of $A$: $[[2, 3\mid 9], [1, -1\mid 3]]$
- Every row operation on $A$ **must also be applied to $B$**

### Three Legal Row Operations
1. **Swap rows:** $R_i \leftrightarrow R_j$ — reorder equations
2. **Scale a row:** $R_i \leftarrow c \cdot R_i$ — multiply equation by constant
3. **Row combination:** $R_i \leftarrow R_i + c \cdot R_j$ — eliminate a variable

### REF vs RREF

| Form | Rules | How to solve |
|---|---|---|
| **REF** | Staircase pattern · zeros below pivots | Back-substitution (bottom up) |
| **RREF** | REF + each pivot = 1 + zeros above pivots | Read answer directly from last column |

**Special cases:**

| What appears | Meaning |
|---|---|
| Row of all zeros `[0 0 0 \| 0]` | Free variable → infinite solutions |
| Contradiction `[0 0 0 \| 5]` | No solution (inconsistent system) |

In [ ]:
import numpy as np

# ============================================================
# MODULE 5  SOLVING AX = B
# ============================================================

# System: 2x + 3y = 9  and  x - y = 3

# --- Augmented matrix [A | b] ---
aug = np.array([[2.0,  3.0,  9.0],
                [1.0, -1.0,  3.0]])
print("Augmented matrix [A|b]:")
print(aug)

# --- Step 1: Scale row 0 → leading element = 1 ---
aug[0] = aug[0] / 2
print("\nStep 1 R0 ← R0/2:")
print(aug)

# --- Step 2: Eliminate x from row 1 ---
aug[1] = aug[1] - aug[0]
print("\nStep 2 R1 ← R1 - R0:  (now in REF)")
print(aug)

# --- Back-substitution from REF ---
y = aug[1, 2] / aug[1, 1]           # from row 1: -2.5y = -1.5
x = (aug[0, 2] - aug[0,1]*y) / aug[0,0]  # substitute into row 0
print(f"\nBack-sub: y = {y},  x = {x}")

# --- Continue to RREF: make pivot of R1 = 1, eliminate above ---
aug[1] /= aug[1, 1]                  # R1 ← R1 / pivot
aug[0] -= aug[0, 1] * aug[1]         # R0 ← R0 - 1.5×R1
print("\nRREF — read solution from last column:")
print(aug)
print(f"x = {aug[0,2]},  y = {aug[1,2]}")

# --- numpy solve (does all of the above automatically) ---
A = np.array([[2.0, 3.0], [1.0, -1.0]])
b = np.array([9.0, 3.0])
sol = np.linalg.solve(A, b)
print(f"\nnp.linalg.solve: x = {sol[0]},  y = {sol[1]}")

# --- 3-variable system ---
A3 = np.array([[1.0, 2,  1],
               [2.0, 1, -1],
               [3.0,-1,  2]])
b3 = np.array([8.0, 1, 3])
sol3 = np.linalg.solve(A3, b3)
print(f"\n3-variable solution [x,y,z]: {sol3.round(4)}")
print(f"Verify A3 @ sol3 = b3: {np.allclose(A3 @ sol3, b3)}")

In [ ]:
import numpy as np

# ============================================================
# 5  SPECIAL CASES: infinite & no solutions
# ============================================================

# --- Case 1: Infinite solutions (free variable) ---
# x + 2y = 4    → both equations are the same scaled
# 2x + 4y = 8
A_inf = np.array([[1.0, 2.0], [2.0, 4.0]])
b_inf = np.array([4.0, 8.0])

rank_A = np.linalg.matrix_rank(A_inf)
rank_Ab = np.linalg.matrix_rank(np.column_stack([A_inf, b_inf]))
print("=== Infinite solutions ===")
print(f"rank(A) = {rank_A},  rank([A|b]) = {rank_Ab}")
print(f"rank < n_vars → free variable → infinite solutions")
# np.linalg.solve will error here
try: np.linalg.solve(A_inf, b_inf)
except np.linalg.LinAlgError as e: print(f"solve() error: {e}")

# --- Case 2: No solution (inconsistent) ---
# x + y = 3
# x + y = 5   → same left side, different right side — impossible
A_no = np.array([[1.0, 1.0], [1.0, 1.0]])
b_no = np.array([3.0, 5.0])

rank_A = np.linalg.matrix_rank(A_no)
rank_Ab = np.linalg.matrix_rank(np.column_stack([A_no, b_no]))
print("\n=== No solution ===")
print(f"rank(A) = {rank_A},  rank([A|b]) = {rank_Ab}")
print(f"rank(A) < rank([A|b]) → inconsistent → no solution")

---
## Module 6 — Rank, Linear Independence & the Null Space

Together these explain **why models crash** and **what "singular matrix" really means**.

### Null Space (Kernel)

The **null space** of $A$ = all vectors $\mathbf{x}$ such that $A\mathbf{x} = \mathbf{0}$

- If null space = {zero vector only} → $A$ is full rank → **inverse exists**
- If null space has non-zero vectors → $A$ is rank-deficient → **no inverse**

> **ML intuition:** If a feature vector lands in the null space of the weight matrix, that feature has **zero influence on any prediction** — it's completely invisible to the model.

### The Four Fundamental Subspaces

| Subspace | What it is | Dimension |
|---|---|---|
| **Column space** | All reachable outputs $A\mathbf{x}$ | = rank |
| **Null space** | All inputs that map to $\mathbf{0}$ | = n − rank |
| **Row space** | Column space of $A^\top$ | = rank |
| **Left null space** | Null space of $A^\top$ | = m − rank |

> For a 200×20 dataset with rank 15: column space is 15D, and **5 of your 20 features are redundant combinations** of the others.

In [ ]:
import numpy as np
from scipy.linalg import null_space

# ============================================================
# MODULE 6  RANK & NULL SPACE
# ============================================================

# --- A rank-deficient matrix ---
A = np.array([[1.0, 2.0, 3.0],
              [2.0, 4.0, 6.0]])   # row 1 = 2 × row 0

print("Matrix A (2×3, but rank 1):")
print(A)
print("Rank:", np.linalg.matrix_rank(A))   # 1 — only 1 independent row

# --- Null space: all x such that Ax = 0 ---
ns = null_space(A)
print("\nNull space basis (columns are null vectors):")
print(ns.round(4))
print("Null space dimension:", ns.shape[1])   # n - rank = 3 - 1 = 2

# Verify: A @ (null space vectors) should be zero
print("A @ null_space (should be ~0):")
print(np.round(A @ ns, 10))

# --- Real-world: redundant feature in the null space ---
np.random.seed(42)
trips     = np.random.rand(100) * 10 + 5       # random trips
fare      = trips * 1.5 + np.random.rand(100)  # correlated fare
fare_x2   = fare * 2                           # EXACT linear multiple → in null space!

X = np.column_stack([trips, fare, fare_x2])
ns_X = null_space(X)
print("\n=== Dataset null space ===")
print(f"Shape: {X.shape},  Rank: {np.linalg.matrix_rank(X)}")
print("Null space basis:", ns_X.T.round(4))
print("→ the [0, 2, -1] direction: 2×fare - fare_x2 = 0 always")

# --- Using the four subspaces to understand a dataset ---
def subspace_analysis(X):
    m, n = X.shape
    rank = np.linalg.matrix_rank(X)
    print(f"Matrix: {m}×{n}  |  rank={rank}")
    print(f"  Column space dim  : {rank}   (reachable outputs)")
    print(f"  Null space dim    : {n-rank} (redundant input directions)")
    print(f"  Row space dim     : {rank}")
    print(f"  Left null space   : {m-rank}")

print("\n=== Subspace Analysis ===")
X_full = np.random.rand(10, 4)
subspace_analysis(X_full)     # full rank

X_dep  = np.column_stack([trips[:10], fare[:10], fare_x2[:10]])
subspace_analysis(X_dep)      # rank-deficient

---
## Module 7 — Eigenvalues & Eigenvectors: The Natural Directions

**Core idea:** When matrix $A$ transforms most vectors, **both direction and length change**.  
But some special vectors **only get scaled** — direction stays the same.

$$A \cdot \mathbf{v} = \lambda \cdot \mathbf{v}$$

- $\mathbf{v}$ = **eigenvector** — the special direction (unchanged by $A$)
- $\lambda$ = **eigenvalue** — the scaling factor

> **Matatu analogy:** A traffic transformation reshuffles all routes in Nairobi. Most routes get rerouted. But some "natural axes" just get scaled — busier or quieter, but same direction. Those axes are the eigenvectors.

| $\lambda$ | Effect on eigenvector |
|---|---|
| $> 1$ | Stretched |
| $0 < \lambda < 1$ | Compressed |
| $< 0$ | Flipped direction then scaled |
| $= 0$ | Collapsed to zero (singular matrix!) |

### Finding Eigenvalues: The Characteristic Equation

Set $\det(A - \lambda I) = 0$ → solve for $\lambda$ → substitute back to find $\mathbf{v}$

**Example: $A = \begin{pmatrix}2&1\\1&2\end{pmatrix}$**

$$\det\begin{pmatrix}2-\lambda & 1 \\ 1 & 2-\lambda\end{pmatrix} = (2-\lambda)^2 - 1 = 0 \Rightarrow \lambda = 3 \text{ or } \lambda = 1$$

- $\lambda = 3$: eigenvector $[1,1]$ — stretched by 3 along the diagonal
- $\lambda = 1$: eigenvector $[1,-1]$ — length unchanged along the anti-diagonal

### Why They Matter in Data Science

| Application | Role of eigenvalues/eigenvectors |
|---|---|
| **PCA** | Eigenvectors of covariance matrix = principal component directions; eigenvalues = variance captured |
| **Google PageRank** | Dominant eigenvector of web link matrix = page rankings |
| **Graph analysis** | Eigenvalues reveal community structure and information spread |
| **Image compression** | Eigenvectors of pixel covariance = "eigenfaces" |
| **Model stability** | Eigenvalues of the Hessian tell if loss function is at a minimum |

In [ ]:
import numpy as np

# ============================================================
# MODULE 7  EIGENVALUES & EIGENVECTORS
# ============================================================

# --- Diagonal matrix: eigenvectors are the axes ---
A_diag = np.array([[2.0, 0.0],
                   [0.0, 3.0]])   # scales x by 2, y by 3

vals, vecs = np.linalg.eig(A_diag)
print("Diagonal matrix — eigenvectors ARE the axes:")
print("Eigenvalues :", vals)           # [2., 3.]
print("Eigenvectors:\n", vecs)        # [[1,0],[0,1]]

# --- More interesting: mixed matrix A = [[2,1],[1,2]] ---
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])

eigenvalues, eigenvectors = np.linalg.eig(A)
print("\n=== A = [[2,1],[1,2]] ===")
print("Eigenvalues :", eigenvalues)
print("Eigenvectors (each column is one eigenvector):")
print(eigenvectors.round(4))
# v1 = [0.707, 0.707] = [1,1] direction, stretched by 3
# v2 = [0.707,-0.707] = [1,-1] direction, unchanged (λ=1)

# --- VERIFY: A @ v = λ × v ---
print("\n=== Verification: A @ v must equal λ × v ===")
for i in range(2):
    v   = eigenvectors[:, i]
    lam = eigenvalues[i]
    print(f"λ={lam:.0f}: A@v = {(A@v).round(4)},  λ×v = {(lam*v).round(4)},  match = {np.allclose(A@v, lam*v)}")

# --- Characteristic equation roots manually ---
# det(A - λI) = (2-λ)^2 - 1 = λ^2 - 4λ + 3 = 0
coeffs = [1, -4, 3]   # polynomial: λ^2 - 4λ + 3
roots = np.roots(coeffs)
print("\nCharacteristic polynomial roots:", roots)   # [3., 1.]

### 7.3 PCA Step by Step — Eigenvalues in Action

PCA reduces $p$ features to $k$ "most important" directions:

1. **Standardise:** subtract column mean, divide by column std
2. **Covariance matrix:** $C = \frac{X^\top X}{n-1}$ — all feature relationships in one matrix
3. **Eigendecompose $C$:** eigenvectors = directions of max variance; eigenvalues = how much variance
4. **Sort** by eigenvalue descending — largest = most informative
5. **Keep top-k** eigenvectors = your $k$ principal components
6. **Project:** $X_{reduced} = X \cdot V_k$

> sklearn's `PCA` uses SVD internally (Module 8.3) — faster and numerically stable. The steps above show what it's doing.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ============================================================
# 7.3  PCA STEP BY STEP
# ============================================================

np.random.seed(42)
fleet = np.random.randn(20, 5)           # 20 matatus, 5 features
feature_names = ["trips","fare","fuel","passengers","distance"]

# Step 1: Standardise (zero mean, unit std per feature)
scaler = StandardScaler()
X = scaler.fit_transform(fleet)

# Step 2: Covariance matrix
C = np.cov(X.T)                          # (5×5) — all 10 unique covariances

# Step 3: Eigendecomposition
vals, vecs = np.linalg.eigh(C)           # eigh for symmetric matrices (more stable)

# Step 4: Sort descending
idx  = np.argsort(vals)[::-1]
vals = vals[idx];  vecs = vecs[:, idx]

variance_explained = vals / vals.sum()
print("Eigenvalues (= variance captured):", vals.round(3))
print("Variance explained per PC        :", variance_explained.round(3))
print("Cumulative variance               :", variance_explained.cumsum().round(3))

# Step 5 & 6: Keep top-2 components and project
k = 2
V2   = vecs[:, :k]              # top-2 eigenvectors
X_2d = X @ V2                   # project onto 2D

print(f"\nOriginal : {X.shape}  →  Reduced : {X_2d.shape}")
print(f"Information retained: {variance_explained[:k].sum()*100:.1f}%")

# === Compare with sklearn ===
pca_sk = PCA(n_components=2)
X_2d_sk = pca_sk.fit_transform(X)
print("\nsklearn variance explained  :", pca_sk.explained_variance_ratio_.round(3))
print("Manual  variance explained   :", variance_explained[:2].round(3))
print("(sklearn uses SVD internally — more stable, same result)")

---
## Module 8 — Matrix Decompositions: LU, QR, SVD

A decomposition writes $A$ as a product of simpler matrices — like factoring $12 = 2 \times 2 \times 3$.  
Each reveals hidden structure and makes specific computations much cheaper.

### 8.1 LU: $A = LU$
- $L$ = lower triangular (zeros above diagonal)
- $U$ = upper triangular (zeros below diagonal)
- Gaussian elimination **stored as matrix multiplication**
- **Use:** solving $AX = B$ when you have many different $B$ vectors for the same $A$
  - Do LU once ($O(n^3)$) → each solve $O(n^2)$ — big saving

### 8.2 QR: $A = QR$
- $Q$ = orthogonal ($Q^\top Q = I$ · columns are perpendicular unit vectors)
- $R$ = upper triangular  
- Key property: $Q^{-1} = Q^\top$ — trivial to compute!
- **Use:** least-squares regression (overdetermined systems) — more stable than forming $X^\top X$

### 8.3 SVD: $A = U \Sigma V^\top$ — The Most Powerful

| Matrix | Shape | What it is |
|---|---|---|
| $U$ | $m \times m$ | Left singular vectors (orthogonal) — output directions |
| $\Sigma$ | $m \times n$ | Diagonal — singular values $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$ |
| $V^\top$ | $n \times n$ | Right singular vectors (orthogonal) — input directions |

- Works on **any matrix** — square or not
- Singular values = how "important" each dimension is
- Set small $\sigma_i \to 0$ → compressed approximation
- **Used in:** PCA · image compression · recommendation systems · pseudoinverse

In [ ]:
import numpy as np
from scipy.linalg import lu, lu_factor, lu_solve

# ============================================================
# 8.1  LU DECOMPOSITION
# ============================================================

A = np.array([[2.0, 1.0, 1.0],
              [4.0, 3.0, 3.0],
              [8.0, 7.0, 9.0]])

P, L, U = lu(A)
print("L (lower triangular):")
print(L.round(3))
print("U (upper triangular):")
print(U.round(3))
print("Verify P @ A == L @ U:", np.allclose(P @ A, L @ U))

# --- LU shines when solving SAME A with multiple b vectors ---
lu_obj = lu_factor(A)   # decompose ONCE — O(n^3)

print("\nSolving same A with 3 different b vectors:")
b_vectors = [np.array([4., 10., 20.]),
             np.array([1., 2., 3.]),
             np.array([0., 5., 15.])]

for b in b_vectors:
    x = lu_solve(lu_obj, b)   # each solve is O(n^2) — reuse decomp
    print(f"  b={b} → x={x.round(4)}")

In [ ]:
import numpy as np
from scipy.linalg import qr, svd as scipy_svd

# ============================================================
# 8.2  QR DECOMPOSITION
# ============================================================

A = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])   # 3×2 (overdetermined — more equations than unknowns)

Q, R = qr(A, mode='economic')   # reduced QR
print("Q (orthogonal, columns are unit vectors):")
print(Q.round(4))
print("R (upper triangular):")
print(R.round(4))
print("Q.T @ Q = I:", np.allclose(Q.T @ Q, np.eye(Q.shape[1])))

# --- Least squares regression via QR ---
# More equations than unknowns → no exact solution → minimise ||Ax - b||^2
X_data = np.array([[1,8],[1,10],[1,12],[1,14],[1,16]], dtype=float)  # [bias, trips]
y_data = np.array([12., 15., 16., 20., 22.])

# lstsq uses QR internally
beta, residuals, rank, sv = np.linalg.lstsq(X_data, y_data, rcond=None)
print(f"\nRegression via lstsq (QR): intercept={beta[0]:.3f}, slope={beta[1]:.3f}")

# ============================================================
# 8.3  SVD — THE MOST POWERFUL
# ============================================================

A = np.array([[1.,2],[3,4],[5,6]])   # 3×2 — non-square

U, s, Vt = scipy_svd(A)
print("\n=== SVD of 3×2 matrix ===")
print("U  shape:", U.shape)    # (3,3)
print("Singular values:", s.round(4))   # [9.5255, 0.5143]
print("Vt shape:", Vt.shape)   # (2,2)

# Reconstruct: Σ is zero-padded to match shape (3×2)
Sigma = np.zeros(A.shape)
Sigma[:len(s), :len(s)] = np.diag(s)
A_back = U @ Sigma @ Vt
print("Reconstruction error:", np.max(np.abs(A - A_back)))   # ~1e-15

# Truncated SVD: rank-1 approximation (compression)
Sigma_1 = np.zeros(A.shape)
Sigma_1[0,0] = s[0]            # keep ONLY the largest singular value
A_rank1 = U @ Sigma_1 @ Vt

print("\nOriginal A:")
print(A)
print("Rank-1 approximation:")
print(A_rank1.round(3))
variance_kept = (s[0]**2) / (s**2).sum()
print(f"Information kept: {variance_kept*100:.1f}%")

---
## Module 9 — Real-World ML Pipelines: Everything Comes Together

### 9.1 Linear Regression: Every Piece Traced

$$\hat{\beta} = (X^\top X)^{-1} X^\top \mathbf{y}$$

| Piece | Module | Role |
|---|---|---|
| $X$ | 2 | Your feature matrix ($m \times n$) |
| $X^\top$ | 2.5 | Transpose — makes dimensions align |
| $X^\top X$ | 2.4 | Matrix multiply → $n \times n$ square matrix |
| $(X^\top X)^{-1}$ | 4 | Inverse — exists only if $X$ has full column rank (Module 6) |
| $X^\top \mathbf{y}$ | 2.4 | Dot products of each feature column with target |

sklearn uses SVD/QR for stability, but this formula is the intent.

### 9.2 Neural Networks: Every Layer Is One Matrix Multiply

For a layer with $p$ neurons processing a batch of $m$ samples:

$$Z = X \cdot W^\top + \mathbf{b} \qquad (m \times p) = (m \times n)(n \times p)$$

- Every **neuron** = dot product (Module 1.3)
- Every **layer** = matrix multiplication (Module 2.4)
- **Backpropagation** = chain rule on those matrix multiplies (transposed weight matrices)
- Without matrix operations, neural networks at any useful scale are impossible

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# ============================================================
# 9.1  LINEAR REGRESSION: MANUAL vs sklearn
# ============================================================

fleet = np.array([
    [8,  12.0, 3.5],   # [trips, fare_k, fuel]
    [10, 15.0, 4.1],
    [12, 16.0, 4.4],
    [14, 20.0, 5.0],
    [16, 22.0, 5.3],
])
X_feat = fleet[:, :2]    # features: trips, fare
y_fuel = fleet[:, 2]     # target: fuel consumption

# --- Manual normal equation: β = (X.T @ X)^{-1} @ X.T @ y ---
X_design = np.hstack([np.ones((5,1)), X_feat])   # add bias column of 1s

XtX  = X_design.T @ X_design          # (3×5) @ (5×3) = 3×3
Xty  = X_design.T @ y_fuel            # (3×5) @ (5,)  = (3,)
beta = np.linalg.solve(XtX, Xty)      # solve instead of inv() — better practice!

print("Manual β [intercept, trips, fare]:", beta.round(4))

# --- sklearn (uses SVD internally) ---
lr = LinearRegression()
lr.fit(X_feat, y_fuel)
print("sklearn  [intercept, trips, fare]:", [lr.intercept_.round(4)] + list(lr.coef_.round(4)))

# Predictions
y_pred_manual = X_design @ beta
y_pred_sklearn = lr.predict(X_feat)
print("\nPredictions match:", np.allclose(y_pred_manual, y_pred_sklearn, atol=1e-6))

print("\n=== Shape trace through the normal equation ===")
print(f"X_design : {X_design.shape}")
print(f"X.T @ X  : {XtX.shape}   ← square, invertible if full rank")
print(f"X.T @ y  : {Xty.shape}    ← right-hand side")
print(f"beta     : {beta.shape}    ← one coefficient per feature")

In [ ]:
import numpy as np

# ============================================================
# 9.2  NEURAL NETWORK FORWARD PASS — ONE LAYER
# ============================================================

np.random.seed(0)

# One batch of fleet data (5 matatus, 3 features)
fleet = np.array([
    [8,  12.0, 3.5],
    [10, 15.0, 4.1],
    [12, 16.0, 4.4],
    [14, 20.0, 5.0],
    [16, 22.0, 5.3],
], dtype=float)   # shape (5, 3) = (batch, features)

m_batch  = fleet.shape[0]   # 5 samples
n_input  = fleet.shape[1]   # 3 input features
p_hidden = 4                # 4 neurons in this hidden layer

# Each neuron has its own weight vector (one row of W)
W = np.random.randn(p_hidden, n_input)   # shape (4, 3)
b = np.zeros(p_hidden)                   # bias: shape (4,)

print("=== Single Layer Forward Pass ===")
print(f"Input X   : {fleet.shape}    (batch × features)")
print(f"Weights W : {W.shape}    (neurons × input_features)")

# One matrix multiply = all neurons, all samples at once
Z = fleet @ W.T + b   # shape (5, 4) = (m_batch × p_hidden)
print(f"Pre-act Z : {Z.shape}    (batch × neurons)  ← ONE matrix multiply")

# Apply ReLU activation element-wise (not a matrix operation — just clipping)
A_out = np.maximum(0, Z)
print(f"After ReLU: {A_out.shape}    (same shape — zeros for negatives)")

print("\nPre-activation Z:")
print(Z.round(3))
print("\nPost-ReLU A:")
print(A_out.round(3))

# This is what pytorch does internally:
# forward = lambda X: np.maximum(0, X @ W.T + b)
print("\n→ A full neural network is just: repeat this for each layer")
print("  Backprop = compute gradients via transposed weight matrices (W.T)")

---
## Quick Reference

| Operation | When you need it | Code |
|---|---|---|
| Dot product | Scoring, similarity, one neuron's output | `np.dot(u, v)` |
| Cosine similarity | Feature similarity (scale-invariant) | `cosine_similarity(X)` (sklearn) |
| Matrix multiply | Batch predictions, layer forward pass, normal equations | `A @ B` |
| Transpose | Dimension alignment, $X^\top X$, gradients | `A.T` |
| Covariance matrix | All feature relationships + PCA input | `np.cov(X.T)` |
| Correlation matrix | All pairwise Pearson r at once | `df.corr()` |
| Determinant | Check invertibility | `np.linalg.det(A)` |
| Matrix rank | Check for redundant/dependent features | `np.linalg.matrix_rank(A)` |
| Solve $AX = B$ | Regression coefficients, equilibrium problems | `np.linalg.solve(A, b)` |
| Null space | Find invisible/redundant directions | `scipy.linalg.null_space(A)` |
| Eigendecomposition | PCA (manual), stability analysis | `np.linalg.eig(A)` |
| SVD | PCA (fast), compression, recommenders | `np.linalg.svd(A)` |
| LU decomposition | Repeated solves with same A | `scipy.linalg.lu(A)` |
| Least squares | Overdetermined regression | `np.linalg.lstsq(X, y)` |

---

> **The single most important takeaway:**  
> Every time a machine learning algorithm processes a batch of data, it is doing **matrix multiplication**.  
> Understanding that one operation — what it computes, why it's fast, when it breaks — will make you a better practitioner than memorising ten algorithms.